In [4]:
import base64
import wmi
from screeninfo import get_monitors

def parse_edid(edid_bytes):
    """Extract monitor brand + model from EDID."""
    # Manufacturer ID (2 bytes)
    manufacturer_id = ((edid_bytes[8] << 8) | edid_bytes[9])
    brand = "".join([
        chr(((manufacturer_id >> 10) & 0x1F) + 64),
        chr(((manufacturer_id >> 5) & 0x1F) + 64),
        chr((manufacturer_id & 0x1F) + 64)
    ])

    # Model name is stored in descriptor blocks
    model = "Unknown Model"
    for i in range(54, 126, 18):
        block = edid_bytes[i:i+18]
        if block[3] == 0xFC:  # Monitor name tag
            text = block[5:18].decode("ascii", errors="ignore").strip()
            if text:
                model = text
                break

    return brand, model


def get_edid_info():
    """Return dict: {display_number: (brand, model)}"""
    c = wmi.WMI(namespace="root\\wmi")
    edid_info = {}

    for idx, monitor in enumerate(c.WmiMonitorDescriptorMethods()):
        try:
            raw = monitor.WmiGetMonitorRawEEdidV1Block(0)[0]
            edid_bytes = bytes(raw)
            brand, model = parse_edid(edid_bytes)
            edid_info[idx] = (brand, model)
        except:
            edid_info[idx] = ("Unknown", "Unknown")

    return edid_info


def list_monitors_with_brand():
    edid_data = get_edid_info()
    monitors = get_monitors()

    for i, m in enumerate(monitors):
        width_px = m.width
        height_px = m.height
        width_mm = m.width_mm
        height_mm = m.height_mm

        # Compute DPI
        if width_mm and height_mm:
            dpi_x = width_px / (width_mm / 25.4)
            dpi_y = height_px / (height_mm / 25.4)
        else:
            dpi_x = dpi_y = None

        brand, model = edid_data.get(i, ("Unknown", "Unknown"))

        print(f"Monitor {i+1}: {m.name}")
        print(f"  Brand: {brand}")
        print(f"  Model: {model}")
        print(f"  Resolution: {width_px} x {height_px}")
        print(f"  Physical size: {width_mm} mm x {height_mm} mm")

        if dpi_x:
            print(f"  DPI X: {dpi_x:.2f}")
            print(f"  DPI Y: {dpi_y:.2f}")
        else:
            print("  DPI: Not reported")

        print()

list_monitors_with_brand()


Monitor 1: \\.\DISPLAY1
  Brand: ACI
  Model: ASUS VT207
  Resolution: 1440 x 900
  Physical size: 408 mm x 255 mm
  DPI X: 89.65
  DPI Y: 89.65

Monitor 2: \\.\DISPLAY2
  Brand: ACI
  Model: ASUS VT207
  Resolution: 1440 x 900
  Physical size: 408 mm x 255 mm
  DPI X: 89.65
  DPI Y: 89.65

Monitor 3: \\.\DISPLAY5
  Brand: DEL
  Model: DELL P1911
  Resolution: 1920 x 1080
  Physical size: 509 mm x 286 mm
  DPI X: 95.81
  DPI Y: 95.92

Monitor 4: \\.\DISPLAY3
  Brand: DEL
  Model: DELL P1911
  Resolution: 900 x 1600
  Physical size: 256 mm x 458 mm
  DPI X: 89.30
  DPI Y: 88.73

Monitor 5: \\.\DISPLAY6
  Brand: DEL
  Model: DELL P2314T
  Resolution: 900 x 1600
  Physical size: 256 mm x 458 mm
  DPI X: 89.30
  DPI Y: 88.73



In [ ]:
## Target the 23" monitor for the charuco calibration

In [7]:
import cv2
import numpy as np
from screeninfo import get_monitors

# -----------------------------
# USER SETTINGS (LANDSCAPE)
# -----------------------------
squares_x = 7   # horizontal squares
squares_y = 5   # vertical squares
square_length_mm = 30
marker_length_mm = 22

# -----------------------------
# Find the only landscape 1920x1080 monitor
# -----------------------------
def find_landscape_monitor():
    monitors = get_monitors()
    candidates = [m for m in monitors if m.width == 1920 and m.height == 1080]
    if len(candidates) != 1:
        raise RuntimeError(f"Expected exactly one 1920x1080 landscape monitor, found {len(candidates)}.")
    return candidates[0]

monitor = find_landscape_monitor()

print(f"Using monitor: {monitor.name}")
print(f"Resolution: {monitor.width} x {monitor.height}")
print(f"Physical size: {monitor.width_mm} mm x {monitor.height_mm} mm")

# -----------------------------
# Compute DPI
# -----------------------------
dpi_x = monitor.width / (monitor.width_mm / 25.4)
dpi_y = monitor.height / (monitor.height_mm / 25.4)
dpi = (dpi_x + dpi_y) / 2

print(f"DPI X: {dpi_x:.2f}, DPI Y: {dpi_y:.2f}, Avg: {dpi:.2f}")

px_per_mm = dpi / 25.4

# -----------------------------
# Generate ChArUco board
# -----------------------------
square_px = int(square_length_mm * px_per_mm)
marker_px = int(marker_length_mm * px_per_mm)

aruco_dict = cv2.aruco.getPredefinedDictionary(cv2.aruco.DICT_4X4_50)
board = cv2.aruco.CharucoBoard(
    (squares_x, squares_y),
    square_px,
    marker_px,
    aruco_dict
)

img = board.generateImage(
    (squares_x * square_px,
     squares_y * square_px)
)

# -----------------------------
# Display fullscreen on selected monitor
# -----------------------------
window_name = "ChArUco Board"
cv2.namedWindow(window_name, cv2.WINDOW_NORMAL)

# Move window to the correct monitor
cv2.moveWindow(window_name, monitor.x, monitor.y)

# Fullscreen
cv2.setWindowProperty(window_name, cv2.WND_PROP_FULLSCREEN, cv2.WINDOW_FULLSCREEN)
cv2.imshow(window_name, img)

print("Press any key to exit.")
cv2.waitKey(0)
cv2.destroyAllWindows()


Using monitor: \\.\DISPLAY5
Resolution: 1920 x 1080
Physical size: 509 mm x 286 mm
DPI X: 95.81, DPI Y: 95.92, Avg: 95.86
Press any key to exit.


In [17]:
import cv2
import numpy as np
from screeninfo import get_monitors

# -----------------------------
# USER SETTINGS (LANDSCAPE BOARD)
# -----------------------------
squares_x = 7   # horizontal squares
squares_y = 5   # vertical squares
square_length_mm = 30
marker_length_mm = 22

# -----------------------------
# Identify the Dell P1911 (1920x1080 landscape)
# -----------------------------
TARGET_NAME = "\\\\.\\DISPLAY5"
TARGET_WIDTH = 1920
TARGET_HEIGHT = 1080

def find_dell_p1911():
    for m in get_monitors():
        if (
            m.name == TARGET_NAME and
            m.width == TARGET_WIDTH and
            m.height == TARGET_HEIGHT
        ):
            return m
    raise RuntimeError("Dell P1911 (DISPLAY5) not found.")

monitor = find_dell_p1911()

print(f"Using monitor: {monitor.name}")
print(f"Resolution: {monitor.width} x {monitor.height}")
print(f"Physical size: {monitor.width_mm} mm x {monitor.height_mm} mm")

# -----------------------------
# Compute pixel pitch (mm per pixel)
# -----------------------------
pixel_pitch_x = monitor.width_mm / monitor.width      # mm per pixel horizontally
pixel_pitch_y = monitor.height_mm / monitor.height    # mm per pixel vertically

print(f"Pixel pitch X: {pixel_pitch_x:.9f} mm/px")
print(f"Pixel pitch Y: {pixel_pitch_y:.9f} mm/px")

# Convert to pixels per mm
px_per_mm_x = 1.0 / pixel_pitch_x
px_per_mm_y = 1.0 / pixel_pitch_y

# Use average for square rendering
px_per_mm = (px_per_mm_x + px_per_mm_y) / 2.0

print(f"Pixels per mm (avg): {px_per_mm:.6f}")

# -----------------------------
# Generate ChArUco board (LANDSCAPE)
# -----------------------------
square_px = int(square_length_mm * px_per_mm)
marker_px = int(marker_length_mm * px_per_mm)

aruco_dict = cv2.aruco.getPredefinedDictionary(cv2.aruco.DICT_4X4_50)
board = cv2.aruco.CharucoBoard(
    (squares_x, squares_y),
    square_px,
    marker_px,
    aruco_dict
)

img = board.generateImage(
    (squares_x * square_px,
     squares_y * square_px)
)

# -----------------------------
# Display fullscreen on Dell P1911
# -----------------------------
window_name = "ChArUco Board"
cv2.namedWindow(window_name, cv2.WINDOW_NORMAL)

# Move window to correct monitor
cv2.moveWindow(window_name, monitor.x, monitor.y)

# Fullscreen
cv2.setWindowProperty(window_name, cv2.WND_PROP_FULLSCREEN, cv2.WINDOW_FULLSCREEN)
cv2.imshow(window_name, img)

print("Press any key to exit.")
cv2.waitKey(0)
cv2.destroyAllWindows()


Using monitor: \\.\DISPLAY5
Resolution: 1920 x 1080
Physical size: 509 mm x 286 mm
Pixel pitch X: 0.265104167 mm/px
Pixel pitch Y: 0.264814815 mm/px
Pixels per mm (avg): 3.774163
Press any key to exit.


In [19]:
import cv2
import numpy as np

# Exact pixel pitch for Dell P2314T
px_pitch_x = 0.26510416   # mm per pixel (horizontal)
px_pitch_y = 0.265203703  # mm per pixel (vertical)

# Landscape-optimized board
squaresX = 12
squaresY = 7

# Choose a clean integer pixel size that fits 1920x1080
square_px = 144                     # pixels per square
marker_px = int(square_px * 0.7)    # 70% of square

# Convert pixel sizes to physical mm (for calibration)
square_mm = square_px * px_pitch_x
marker_mm = marker_px * px_pitch_x

print("Square size (mm):", square_mm)
print("Marker size (mm):", marker_mm)

# Create ChArUco board
aruco_dict = cv2.aruco.getPredefinedDictionary(cv2.aruco.DICT_4X4_50)
board = cv2.aruco.CharucoBoard(
    (squaresX, squaresY),
    square_mm,
    marker_mm,
    aruco_dict
)

# Render at exact pixel size
img_width  = squaresX * square_px
img_height = squaresY * square_px

img = board.generateImage((img_width, img_height))

# Save
cv2.imwrite(f"charuco_x{img_width}_y{img_height}.png", img)

print(f"Saved charuco_x{img_width}_y{img_height}.png at", img_width, "x", img_height, "pixels")


Square size (mm): 38.174999039999996
Marker size (mm): 26.510416
Saved charuco_x1728_y1008.png at 1728 x 1008 pixels
